In [74]:
import torch;
import torch.nn as nn
import torch.utils
import matplotlib.pyplot as plt
import math

In [60]:
import pandas as pd
from datasets import load_dataset
from transformers import AutoTokenizer
import numpy as np

dataset = load_dataset("coastalcph/tydi_xor_rc")
df_train = dataset["train"].to_pandas()
df_validation = dataset["validation"].to_pandas()

langlst = ["ar", "ko", "te"]
df_train_filtered = df_train[df_train["lang"].isin(langlst)]
df_validation_filtered = df_validation[df_validation["lang"].isin(langlst)]

tokenizer = AutoTokenizer.from_pretrained(
    "bert-base-multilingual-cased", use_fast=True
    # "bert-base-multilingual-cased"
)



df_train_ko = df_train[(df_train['lang'] == "ko")]
df_train_ar = df_train[(df_train['lang'] == "ar")]
df_train_te = df_train[(df_train['lang'] == "te")]


## Models chosen:

- 1 n-gram (ask about degrees of n) 

- 2

In [61]:
from nltk import trigrams
from collections import defaultdict
from collections import Counter

def tokenizeQuestion(df):
    questions = df["question"]
    tokenized_questions = []

    for question in questions:
        tokenized_questions.append(tokenizer.tokenize(question))

    return tokenized_questions

t_questions = tokenizeQuestion(df_train_ko)


#Token with frequency 
def buildVocabulary(tokens): 
    tokens_flat = [x for sublist in tokens for x in sublist]
    vocab = Counter(tokens_flat)

    return dict(vocab)

vocabulary = buildVocabulary(t_questions)

vocabulary


{'30': 4,
 '##년': 126,
 '전쟁': 27,
 '##의': 434,
 '승': 13,
 '##자는': 67,
 '누': 352,
 '##구': 371,
 '##인': 1495,
 '##가': 2219,
 '?': 2420,
 '엑': 2,
 '##스': 114,
 '##선': 6,
 '##은': 949,
 '발': 93,
 '##견': 25,
 '##하': 104,
 '##였': 95,
 '##는': 988,
 '아': 74,
 '##테': 15,
 '##네': 24,
 '##에서': 482,
 '언': 477,
 '##제': 461,
 '가장': 529,
 '최': 25,
 '##근': 5,
 '올림픽': 13,
 '##이': 308,
 '올': 8,
 '##렸': 3,
 '##나': 429,
 '##요': 758,
 '세': 183,
 '##상': 219,
 '오': 84,
 '##래': 60,
 '##된': 94,
 '방송': 10,
 '##사는': 42,
 '무': 632,
 '##엇': 609,
 '팔': 8,
 '##레스': 11,
 '##타': 35,
 '수도': 41,
 '어': 373,
 '##딘': 24,
 '별': 20,
 '##자': 55,
 '##리': 116,
 '중': 39,
 '많은': 79,
 '##로': 89,
 '이': 88,
 '##루': 35,
 '##어진': 4,
 '##리는': 40,
 '큰': 198,
 '풍': 2,
 '##력': 16,
 '에': 39,
 '##너': 24,
 '##지': 67,
 '##전': 79,
 '##소': 58,
 '루': 13,
 '15': 2,
 '##세의': 5,
 '본': 13,
 '##명은': 20,
 '컬': 3,
 '##러': 13,
 '텔레비전': 1,
 '처음': 81,
 '출': 43,
 '##시': 90,
 '기': 82,
 '##업': 21,
 '##디': 357,
 '마': 37,
 '##우': 23,
 '##리아': 37,
 '제': 75,
 '##

In [62]:
# Filterd the token with frequency thredshold of count >4 
# To minimized variance when frequency =1 
# Returning the filtered vocab det with tokens of "<s>", "</s>", "<unk>"
# Percentage of selected token in training corpus 
def buildFilteredVocab(freqDict, minCount=4):
    filtered_freq_dict = {tok: c for tok, c in freqDict.items() if c >= minCount}

    #selected token based on count >=4 
    vocab = set(filtered_freq_dict.keys())

    total_sum_voc =  sum(filtered_freq_dict.values()) 
    total_sum_freq = sum(freqDict.values())

    occurenceCoverPerc = (total_sum_voc / total_sum_freq * 100) if total_sum_freq > 0 else 0.0

    #update with token for sencence start, end and unknow token. 
    vocab.update(["<s>", "</s>", "<unk>"])
    return vocab,occurenceCoverPerc

vocab, precentage = buildFilteredVocab(vocabulary)

In [ ]:
# Subistitute the token in the questions
# if they are not in the vocabulary 
# with "<unk>" 

def applyUnk(tokenizedQuestions, vocab):
    return [[tok if tok in vocab else "<unk>" for tok in q] for q in tokenizedQuestions]

t_questions_filtered =applyUnk(t_questions,vocab)
t_questions_filtered

[['30', '##년', '전쟁', '##의', '승', '##자는', '누', '##구', '##인', '##가', '?'],
 ['<unk>',
  '##스',
  '##선',
  '##은',
  '누',
  '##가',
  '발',
  '##견',
  '##하',
  '##였',
  '##는',
  '##가',
  '?'],
 ['아',
  '##테',
  '##네',
  '##에서',
  '언',
  '##제',
  '가장',
  '최',
  '##근',
  '##의',
  '올림픽',
  '##이',
  '올',
  '<unk>',
  '##나',
  '##요',
  '?'],
 ['세',
  '##상',
  '##에서',
  '가장',
  '오',
  '##래',
  '##된',
  '방송',
  '##사는',
  '무',
  '##엇',
  '##인',
  '##가',
  '?'],
 ['팔', '##레스', '##타', '##인', '수도', '##는', '어', '##딘', '##가', '##요', '?'],
 ['별',
  '##자',
  '##리',
  '중',
  '가장',
  '많은',
  '별',
  '##로',
  '이',
  '##루',
  '##어진',
  '별',
  '##자',
  '##리는',
  '무',
  '##엇',
  '##인',
  '##가',
  '?'],
 ['세',
  '##상',
  '##에서',
  '가장',
  '큰',
  '<unk>',
  '##력',
  '에',
  '##너',
  '##지',
  '발',
  '##전',
  '##소',
  '##는',
  '무',
  '##엇',
  '##인',
  '##가',
  '?'],
 ['루', '##이', '<unk>', '##세의', '본', '##명은', '무', '##엇', '##인', '##가', '?'],
 ['<unk>',
  '##러',
  '<unk>',
  '##이',
  '처음',
  '출',
  '##시',
  '##된',
  '기'

In [64]:
# Token to unique ID for NN embedding matrix 
# Build ones and reuse it 
def buildWordToIx(vocab):
    return {tok: i for i, tok in enumerate(sorted(vocab))}


CONTEXT_SIZE = 2  # a "neural trigram" -- same context span as order-3 n-gram

# replace tokens with their ID and padd the sentences start, ends. 
def sentenceToIds(tokens, wordToIx, contextSize=CONTEXT_SIZE):
    padded = ["<s>"] * contextSize + tokens + ["</s>"] 
    return [wordToIx[tok] for tok in padded]

In [71]:
def buildContextTargetPairs(tokenizedQuestionsUnk, wordToIx, contextSize=CONTEXT_SIZE):
    """Slide a fixed window across each id-converted sentence -- same idea
    as n-gram counting, but producing (context, target) TRAINING EXAMPLES
    for a network instead of raw counts."""
    pairs = []
    for tokens in tokenizedQuestionsUnk:
        ids = sentenceToIds(tokens, wordToIx, contextSize)
        for i in range(contextSize, len(ids)):
            pairs.append((ids[i - contextSize:i], ids[i]))
    return pairs
 

## FF NN Model

In [72]:
class FFLM(nn.Module):
    def __init__(self, vocab_size, context_size=CONTEXT_SIZE, embed_dim=16, hidden_dim=32):
        super().__init__()
        self.context_size = context_size
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.hidden = nn.Linear(context_size * embed_dim, hidden_dim)
        self.output = nn.Linear(hidden_dim, vocab_size)


    def forward(self, context_ids):
        # context_ids: (batch, context_size) -> logits: (batch, vocab_size)
        embedded = self.embedding(context_ids)            # (batch, context_size, embed_dim)
        flattened = embedded.reshape(embedded.size(0), -1)
        hidden = torch.tanh(self.hidden(flattened))
        return self.output(hidden)

In [73]:
def trainFeedforwardLm(model, pairs, epochs=300, lr=0.01):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    lossFn = nn.CrossEntropyLoss()
    contexts = torch.tensor([p[0] for p in pairs])
    targets = torch.tensor([p[1] for p in pairs])
 
    for epoch in range(epochs):
        optimizer.zero_grad()
        logits = model(contexts)
        loss = lossFn(logits, targets)
        loss.backward()
        optimizer.step()
        if (epoch + 1) % 100 == 0:
            print(f"  epoch {epoch+1:>4}  loss = {loss.item():.3f}")
 

In [ ]:
@torch.no_grad() #computation efficiency 
def corpusPerplexityFeedforward(model, tokenizedQuestionsUnk, wordToIx, contextSize=CONTEXT_SIZE):
    model.eval()
    pairs = buildContextTargetPairs(tokenizedQuestionsUnk, wordToIx, contextSize)
    contexts = torch.tensor([p[0] for p in pairs])
    targets = torch.tensor([p[1] for p in pairs])
 
    logits = model(contexts)
    logProbs = torch.log_softmax(logits, dim=-1)
    tokenLogProbs = logProbs[range(len(targets)), targets]
 
    model.train()
    return math.exp(-tokenLogProbs.sum().item() / len(targets))
 

In [81]:
 
df_val_ko = df_validation[(df_validation['lang'] == "ko")]
val_unk = applyUnk(tokenizeQuestion(df_val_ko), vocab)
wordToIx = buildWordToIx(vocab)

torch.manual_seed(0)
model = FFLM(vocab_size=len(vocab))
pairs = buildContextTargetPairs(t_questions_filtered, wordToIx)

print(f"Training feedforward LM (context size={CONTEXT_SIZE})...")
trainFeedforwardLm(model, pairs, epochs=300, lr=0.01)

ppl = corpusPerplexityFeedforward(model, val_unk, wordToIx)
print(f"\nFeedforward LM validation perplexity: {ppl:.2f}")

Training feedforward LM (context size=2)...
  epoch  100  loss = 2.514
  epoch  200  loss = 2.003
  epoch  300  loss = 1.777

Feedforward LM validation perplexity: 25.44


In [65]:
len(t_questions)

2422

In [66]:
len(t_questions_filtered)

2422

In [79]:
len(df_val_ko)

356